In [0]:
%run ./config

In [0]:
BRONZE_POPULATION = f"{CATALOG}.{SCHEMA}.brz_population"

In [0]:
population_json_schema = StructType([StructField("countryiso3code", StringType()),
                                     StructField("date", StringType()),
                                     StructField("value", LongType()),
                                     StructField("country", StructType([StructField("id", StringType()),
                                                                        StructField("value", StringType())]))])


In [0]:

population = (spark.table(BRONZE_POPULATION)
    .withColumn("json_data",F.from_json("payload", population_json_schema))
    .select(F.upper(F.trim("json_data.countryiso3code")).alias("country_iso3"),
            F.trim("json_data.country.value").alias("country_name"),
            F.col("json_data.date").cast("int").alias("year"),
            F.col("json_data.value").cast("long").alias("population"),"ingestion_id","collected_at")
    .filter(F.length("country_iso3") == 3)
    .filter(F.col("population").isNotNull())
    .dropDuplicates(["country_iso3", "year"]))

In [0]:
population.write.format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema", "true")\
                .saveAsTable(f"{CATALOG}.{SCHEMA}.slv_population")

In [0]:
%sql
SELECT * FROM mvp_eng_dados.mvp_cancer.slv_population